**The content of $n$-dimensional unit balls from $n=1$ to $n=12$.**
Previous notebooks computed the content (volume) of unit balls in 2D, 3D, and 4D.
This notebook sweeps all dimensions from 1 to 12 in a single loop and
plots the results against the analytic formula:

$$V_n = \frac{\pi^{n/2}}{\Gamma\!\left(\frac{n}{2}+1\right)}$$

The most striking result: **the $n$-ball volume does not grow forever**.
It peaks near $n \approx 5.26$ at $V \approx 5.278$, then decreases toward zero
as $n \to \infty$.
Intuitively, as dimension grows the ball occupies an ever-shrinking fraction
of its bounding hypercube - the corners of the hypercube dominate,
and the ball retreats to the center.

**Efficient accumulation trick.**
Rather than allocating a new coordinate array for each dimension,
a single array `d` accumulates the sum of squared coordinates as each new
dimension is added:
after $k$ dimensions, `d[i]` $= x_1^2 + x_2^2 + \cdots + x_k^2$.
Checking `sqrt(d) <= 1` at each step gives the count inside the $k$-ball
using the same set of random points throughout.

`sympy.prime(k)` supplies the $k$-th prime as the Halton base for dimension $k$,
ensuring all dimensions are independent.

**Halton sequence (Numba-compiled).**
Same implementation as the sphere and hypersphere notebooks.

In [ ]:
"""mc_high_dimensions.ipynb"""

# Cell 01 - Numba-compiled Halton quasi-random sequence

import matplotlib.pyplot as plt
import numpy as np
import sympy
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from numba import float64, int64, vectorize
from scipy.signal import find_peaks
from scipy.special import gamma


@vectorize([float64(int64, int64)], nopython=True)
def halton(n, p):
    h, f = 0, 1
    while n > 0:
        f = f / p
        h += (n % p) * f
        n = int(n / p)
    return h


print(halton(np.int64(10_000), np.int64(2)))

**Sample count.**
6,250,000 points - the same as the 4D hypersphere notebook.
The same set of points is reused across all 12 dimensions by the
accumulation trick in Cell 03.

In [ ]:
# Cell 02 - Number of sample points

total_dots = 6_250_000
print(f"{total_dots = :,}")

**Estimating $V_n$ for $n = 1$ to $12$.**
The accumulation loop adds one coordinate dimension at a time.
At step `dim`, the $k$-th prime (`sympy.prime(dim)`) is the Halton base,
and `halton(...)*2 - 1` maps values to $[-1, 1)$.
After adding the new squared coordinate to `d`, the count of points
with $\sqrt{d} \leq 1$ gives the Monte Carlo estimate for that dimension.
The bounding hypercube multiplier is $2^{\text{dim}}$.

`est[0] = 1` by convention: the 0-ball is a single point with content 1.
`est[1]` is computed by the loop (all 6.25M points in $[-1,1)$ satisfy
$|x| < 1$, so `est[1]` correctly evaluates to 2.0).

In [ ]:
# Cell 03 - Estimate n-ball content for each dimension via accumulated d
# d[i] accumulates x1^2 + x2^2 + ... + xk^2 as each dimension is added
# sympy.prime(dim) gives the k-th prime as the Halton base for dimension k

dimensions = 13  # compute dimensions 0 through 12
n_arr = np.arange(total_dots, dtype=np.int64)
d = np.zeros(total_dots)
est = np.zeros(dimensions)
est[0] = 1  # 0-ball: a single point, content = 1 by convention

for dim in range(1, dimensions):
    print(f"Calculating content of unit {dim:>2}-ball ...")
    p = np.full(total_dots, sympy.prime(dim), dtype=np.int64)
    v = halton(n_arr, p) * 2 - 1  # coordinate in [-1, 1)
    d += v**2  # accumulate sum of squares
    est[dim] = 2**dim * np.count_nonzero(np.sqrt(d) <= 1.0) / total_dots

**Analytic solution and the peak dimension.**
The analytic content $V_n = \pi^{n/2}/\Gamma(n/2+1)$ is evaluated on a
fine grid over $[0, 12]$ (treating $n$ as continuous) to find the
exact peak using `scipy.signal.find_peaks`.
The peak occurs near $n \approx 5.26$, between the 5-ball and 6-ball.

In [ ]:
# Cell 04 - Analytic n-ball volume curve and peak dimension

act_x = np.linspace(0, dimensions - 1, 1000)
act_y = np.power(np.pi, act_x / 2) / gamma(act_x / 2 + 1)
m = find_peaks(act_y)[0][0]  # index of the peak

print(f"Peak dimension : {act_x[m]:.4f}")
print(f"Peak content   : {act_y[m]:.6f}")
print()
print("Analytic values at integer dimensions:")
for n in range(1, dimensions):
    V_n = np.pi ** (n / 2) / gamma(n / 2 + 1)
    print(f"  V_{n:>2} = {V_n:.6f}   MC estimate = {est[n]:.6f}")

**Plotting estimated vs analytic content.**
Blue dots are the Monte Carlo estimates at integer dimensions 0-12.
The red curve is the analytic formula evaluated continuously.
The green dot and vertical line mark the peak at $n \approx 5.26$,
where the unit ball reaches its maximum content before shrinking
toward zero as dimension grows.

In [ ]:
# Cell 05 - Plot Monte Carlo estimates vs analytic n-ball content curve

plt.figure("mc_high_dimensions", figsize=(10, 6))
plt.scatter(
    np.arange(dimensions), est, color="blue", zorder=5, label="Monte Carlo estimate"
)
plt.plot(act_x, act_y, color="red", label="Analytic $V_n$")
plt.scatter(
    act_x[m],
    act_y[m],
    marker="o",
    color="green",
    zorder=6,
    label=f"Peak: $n={act_x[m]:.2f}$, $V={act_y[m]:.3f}$",
)
plt.vlines(act_x[m], 0, act_y[m], color="green", linestyle="--")
plt.title("Content of Unit $n$-Balls ($n = 0$ to $12$)")
plt.xlabel("Dimension $n$")
plt.ylabel("Content $V_n$")
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(1))
ax.xaxis.set_minor_locator(MultipleLocator(0.5))
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()